# Week 3 - In Class Activity: Spatial Autocorrelation with the Meuse Dataset 

**Goal:** Open, explore, model, and diagnose spatial autocorrelation in a point dataset.

In this activity, we use the Meuse dataset, which contains point observations of topsoil heavy-metal concentrations from a floodplain along the Meuse River near Stein, Netherlands. The data include spatial coordinates, heavy-metal concentrations such as zinc, cadmium, copper, and lead, plus environmental covariates such as distance to river, soil type, and flooding frequency. For context, contaminated sediment is transported by the river and tends to be deposited closer to the river bank and in lower-elevation floodplain areas.

The central question is: **after fitting a simple non-spatial regression model, do the residuals still show spatial patterning?** If they do, then the model has not fully explained the spatial structure in zinc concentrations.

## 1. Setup

This cell imports the Python packages used in the workbook.

Most of the data handling and regression modeling is done with `pandas`, `geopandas`, and `scikit-learn`. The spatial diagnostics use specialized spatial libraries because Moran's I, spatial weights, and variograms are not part of core `scikit-learn`.

- `pandas` stores tabular data.
- `geopandas` stores spatial data with a geometry column.
- `scikit-learn` builds the preprocessing and regression pipeline.
- `libpysal` creates spatial neighbor weights.
- `esda` calculates Moran's I.
- `scikit-gstat` calculates the variogram.
- `contextily` adds an OpenStreetMap basemap.

In [ ]:
# !pip install pandas geopandas scikit-learn libpysal esda scikit-gstat contextily matplotlib

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

from pandas.plotting import scatter_matrix

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import root_mean_squared_error, r2_score

from libpysal.weights import KNN
from esda.moran import Moran
from skgstat import Variogram

## 2. Load the Meuse data

This cell reads the Meuse data and converts it into a `GeoDataFrame`.

The original `x` and `y` coordinates are in **EPSG:28992**, also called the Dutch national grid / RD New coordinate system. Defining the coordinate reference system matters because later we reproject the data to Web Mercator (`EPSG:3857`) so it lines up correctly with OpenStreetMap tiles.

In [ ]:
# load from the course GitHub repository
meuse = pd.read_csv("https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/meuse.csv")
meuse = gpd.GeoDataFrame(
        meuse,
        geometry=gpd.points_from_xy(meuse["x"], meuse["y"]),
        crs="EPSG:28992",
    )

### What to notice

After loading the data, we should have one row per soil sample. The `geometry` column stores the point location for each sample. The variables `x` and `y` are coordinates, while variables such as `zinc`, `cadmium`, `copper`, and `lead` are measured concentrations at those locations.

## 3. Data exploration

Before fitting a model, we explore the dataset so we understand what each variable represents, whether values are missing, and whether the target variable appears skewed or spatially clustered.

This is an important step because spatial data often violate the assumption that observations are independent. Nearby locations may have similar values because they share environmental conditions, transport processes, land use history, or sampling design.

In [ ]:
meuse

### Dataset structure

This summary tells us how many observations and variables are present, what the column names are, which variables are numeric or categorical, and whether there are missing values.

Missing values matter because many modeling functions cannot use rows with missing predictors. In the modeling section, the `scikit-learn` pipeline includes imputation steps so the workflow is explicit and reproducible.

In [ ]:
# Print common metrics for dataframes

print("Shape:", meuse.shape)
print("\nColumns:")
print(meuse.columns.tolist())
print("\nData types:")
print(meuse.dtypes)
print("\nMissing values per column:")
print(meuse.isna().sum().sort_values(ascending=False))

### Numeric summaries

The table below reports summary statistics for numeric variables. Pay close attention to the heavy-metal variables such as `zinc`, `cadmium`, `copper`, and `lead`.

Environmental concentration data are often right-skewed: a few locations may have much higher concentrations than most locations. That is one reason we later model `log_zinc` instead of raw `zinc`.


In [ ]:
# TODO: get the numberic summaries of each column including mean, median, count, min, max, std, and quantiles

### Pairwise relationships

The scatter matrix lets us compare zinc to distance from the river and to other metal concentrations.

Typical patterns to look for:

- Strong positive relationships among heavy metals may suggest a shared contamination process.
- A relationship between `dist` and `zinc` may suggest that contamination is related to distance from the river.
- Curved or fan-shaped relationships may suggest that a transformation, such as a logarithm, is useful.

In [ ]:
pair_vars = ["zinc", "dist", "cadmium", "copper", "lead"]
scatter_matrix(meuse[pair_vars].dropna(), figsize=(10, 10), diagonal="hist")
plt.suptitle("Scatter matrix of selected variables")
plt.tight_layout()
plt.show()

### Spatial view over OpenStreetMap

This map places the sample points over an OpenStreetMap basemap and colors them by zinc concentration.

Before plotting with a web basemap, the data are reprojected to `EPSG:3857`, the coordinate system used by most web map tiles. The map helps to connect the numerical results to the geography of the study area.

Interpretation guide:

- Points with similar colors near one another suggest spatial clustering.
- Higher zinc values concentrated in one part of the floodplain suggest that location matters.
- If high values occur near the river or flood-prone areas, that supports the idea that river transport and deposition may influence contamination.


In [ ]:
# Reproject to Web Mercator for OpenStreetMap tiles
gdf_3857 = meuse.to_crs(epsg=3857)

# Plot zinc values on top of an OpenStreetMap basemap
fig, ax = plt.subplots(figsize=(10, 10))
gdf_3857.plot(
    ax=ax,
    column="zinc",
    cmap="viridis",
    markersize=40,
    legend=True,
    alpha=0.85
)

cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.set_axis_off()
ax.set_title("Meuse sample points over OpenStreetMap")
plt.tight_layout()
plt.show()

## 4. Model preprocessing and regression

We now build a simple non-spatial model for zinc concentration.

The response variable is `log_zinc`, the natural logarithm of zinc. This transformation is common for concentration data because it reduces right-skewness and makes large values less dominant in the regression.

The predictors are:

- `dist`: distance to the river, treated as numeric.
- `soil`: soil class, treated as categorical.
- `ffreq`: flooding frequency class, treated as categorical.

This model is intentionally simple. It asks: **how much of the zinc pattern can be explained by basic environmental covariates before explicitly modeling spatial dependence?**


In [ ]:
# Response transformation
df = meuse.copy()
df["log_zinc"] = np.log(df["zinc"])

# Predictors from the original example
feature_cols = ["dist", "soil", "ffreq"]

X = df[feature_cols]
y = df["log_zinc"]

### Build the `scikit-learn` preprocessing pipeline

The pipeline separates numeric and categorical variables.

For the numeric variable `dist`, the workflow imputes missing values with the median and standardizes the values. Standardization is not required for ordinary linear regression, but it is useful practice in many machine-learning workflows.

For categorical variables `soil` and `ffreq`, the workflow fills missing values with the most common category and then one-hot encodes the categories. `drop="first"` removes one category from each categorical variable so the regression has a reference category.


In [ ]:
# Treat soil and ffreq as categorical.
# dist is numeric.
categorical_features = ["soil", "ffreq"]
numeric_features = ["dist"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])


### Fit the model and calculate residuals

After fitting the model, we calculate two quantities:

- `pred_m1`: the model's predicted log zinc value.
- `resid_m1`: the residual, calculated as observed log zinc minus predicted log zinc.

Residuals are central to the spatial diagnostic step. If the model fully explains the systematic pattern in zinc, the residuals should look roughly random in space. If residuals are still spatially clustered, the model is missing spatial structure.


In [ ]:
# Fit on the full dataset
model.fit(X, y)

# Residuals
df["pred_m1"] = model.predict(X)
df["resid_m1"] = df["log_zinc"] - df["pred_m1"]

print("\nModel fit on full data")
print("R^2:", r2_score(y, df["pred_m1"]))
print("RMSE:", root_mean_squared_error(y, df["pred_m1"]))

### Interpreting the model fit

`R^2` measures the proportion of variation in `log_zinc` explained by the predictors. Larger values mean the model explains more variation, but `R^2` alone does not tell us whether the model assumptions are satisfied.

`RMSE` measures the typical prediction error on the log-zinc scale. Smaller values indicate better fit.

Important caution: these metrics are calculated after fitting and predicting on the same dataset. They describe in-sample fit, not out-of-sample predictive performance. That is why we also use cross-validation below.


### Cross-validation

Cross-validation gives a more realistic estimate of predictive performance. The dataset is split into five parts. The model is trained on four parts and tested on the remaining part, repeated five times.

The mean cross-validated `R^2` is usually lower than the in-sample `R^2`. If it is much lower, the model may not generalize well.

In [ ]:
# train/test split for teaching purposes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_cv = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

cv_scores = cross_val_score(model_cv, X, y, cv=5, scoring="r2")
print("\n5-fold CV R^2 scores:", cv_scores)
print("Mean CV R^2:", cv_scores.mean())

## 5. Spatial diagnostics: neighborhood weights

To measure spatial autocorrelation, we first define which observations are neighbors.

Here, each sample point is connected to its six nearest neighboring points using k-nearest neighbors. The weights are row-standardized, meaning each point's neighbor weights sum to one.

This step matters because Moran's I depends on the neighborhood definition. A different number of neighbors could produce slightly different results.

In [ ]:
coords = np.column_stack([df["x"].values, df["y"].values])

# k-nearest-neighbor weights
knn = KNN.from_array(coords, k=6)
knn.transform = "R"  # row-standardized, values sume to 1

## 6. Moran's I

Moran's I measures whether similar values occur near one another.

Interpretation guide:

- Positive Moran's I: nearby observations tend to have similar values, indicating clustering.
- Moran's I near zero: little or no spatial autocorrelation.
- Negative Moran's I: nearby observations tend to be dissimilar, indicating spatial dispersion or checkerboard-like patterning.
- Small p-value, often below 0.05: the observed spatial pattern is unlikely under random spatial arrangement.

We calculate Moran's I twice:

1. For `log_zinc`, to test whether zinc itself is spatially clustered.
2. For `resid_m1`, to test whether the regression residuals are still spatially clustered after accounting for distance, soil, and flooding frequency.

In [ ]:
# Moran's I on raw target
moran_raw = Moran(df["log_zinc"].values, knn)
print("\nMoran's I on log_zinc")
print("I:", moran_raw.I)
print("p-value:", moran_raw.p_sim)

# Moran's I on residuals
moran_resid = Moran(df["resid_m1"].values, knn)
print("\nMoran's I on residuals")
print("I:", moran_resid.I)
print("p-value:", moran_resid.p_sim)

### Interpreting the Moran's I results

If `log_zinc` has a positive and statistically significant Moran's I, zinc concentrations are spatially clustered. That means nearby samples tend to have similar zinc levels.

The residual Moran's I is the more important diagnostic for the regression model:

- If residual Moran's I is not significant, then the covariates may have explained most of the spatial structure relevant to zinc.
- If residual Moran's I is still positive and significant, then the model has missed spatial structure. This suggests that nearby residuals are similar and that a spatial modeling approach may be needed.

In practical terms, significant residual spatial autocorrelation means the simple linear model may overstate how much independent information is present in the data. It can also produce overly optimistic uncertainty estimates because nearby observations are not fully independent.


## 7. Variogram of residuals

A variogram describes how dissimilarity changes with distance. Here, we calculate a variogram for the model residuals.

The y-axis shows semivariance, which is a measure of how different residuals are from one another. The x-axis shows distance between sample pairs.

Interpretation guide:

- If semivariance increases with distance, nearby residuals are more similar than distant residuals. This indicates spatial dependence.
- If the curve levels off, the distance where it levels off is related to the spatial range of dependence.
- A flat variogram suggests little spatial structure in the residuals.
- A high semivariance at very short distances may suggest measurement error or microscale variation

In [ ]:
vgm_resid = Variogram(
    coords,
    df["resid_m1"].values,
    normalize=False
)

fig, ax = plt.subplots(figsize=(7,5))

ax.scatter(vgm_resid.bins, vgm_resid.experimental, s=50, label="Experimental")
ax.plot(vgm_resid.bins, vgm_resid.fitted_model(vgm_resid.bins),
        label="Fitted Model")

ax.set_xlabel("Distance")
ax.set_ylabel("Semivariance")
ax.set_title("Empirical Variogram of Residuals")
ax.legend()

plt.show()

### Interpreting the variogram results

The variogram complements Moran's I. Moran's I gives a single summary of spatial autocorrelation for a chosen neighborhood structure, while the variogram shows how spatial dependence changes over distance.

If the residual variogram rises with distance before flattening out, the residuals still contain spatial structure. That means the simple non-spatial regression has not captured all of the geographic process affecting zinc.

If the residual variogram is nearly flat, the residuals are not strongly distance-dependent. That would support the idea that the predictors explain most of the spatial pattern.

Together, the Moran's I and variogram results help answer the main question of the workbook: **does a simple non-spatial model remove the spatial structure from zinc concentrations, or is a spatial model still needed?**


## 8. Overall discussion

This workflow shows a common sequence in spatial data analysis:

1. **Explore the data.** Understand the variables, distributions, missing values, and spatial layout.
2. **Fit a baseline non-spatial model.** Use environmental covariates such as distance to river, soil type, and flooding frequency to explain zinc concentration.
3. **Examine residuals.** Residuals show what the model did not explain.
4. **Test for spatial autocorrelation.** Moran's I asks whether similar residuals occur near one another.
5. **Examine spatial dependence by distance.** The variogram shows whether residual similarity changes as locations get farther apart.

If both Moran's I and the variogram suggest residual spatial structure, then a more advanced model may be appropriate. Possible next steps include adding more predictors, using spatial lag or spatial error models, fitting a geostatistical model, or using kriging to model spatially structured residual variation.

The most important lesson is that strong model performance metrics do not automatically mean the model is spatially appropriate. Spatial diagnostics are needed because geographic observations are often not independent.


## 9. Reflection questions

1. Why might zinc concentrations be higher in some parts of the floodplain than others?
2. Why did we model `log_zinc` instead of raw `zinc`?
3. What does a positive Moran's I tell us about the spatial arrangement of values?
4. Why do we check Moran's I on the residuals, not only on the original zinc values?
5. If residual spatial autocorrelation remains, what does that imply about the regression model?
6. How does the variogram add information beyond Moran's I?